<a href="https://colab.research.google.com/github/saish-res-phd/GenerativeAIwindTurbine/blob/main/Windturbine_Dlinear_Tmixer_TimesNet.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>



---


**Implememting the liabrary on wind turbine dataset**

In this we will implement the liabrary for PatchTST

Liabrary link:https://github.com/thuml/Time-Series-Library


---







---


**Import Data**

> The data which is used in this project is from kaggle. In Wind Turbines, Scada Systems measure and save data's like wind speed, wind direction, generated power etc. for 10 minutes intervals. This file was taken from a wind turbine's scada system that is working and generating power in Turkey.

Link:https://www.kaggle.com/datasets/berkerisen/wind-turbine-scada-dataset/data






In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from google.colab import drive
drive.mount('/content/drive', force_remount=True)

# Specify the file paths
folder_path = '/content/drive/MyDrive/GenerativeAI/'

windTurbdf = pd.read_csv(folder_path + 'windturbine.csv')

windTurbdf

Mounted at /content/drive


,Date/Time,LV ActivePower (kW),Wind Speed (m/s),Theoretical_Power_Curve (KWh),Wind Direction (°)
0,01 01 2018 00:00,380.047791,5.311336,416.328908,259.994904
1,01 01 2018 00:10,453.769196,5.672167,519.917511,268.641113
2,01 01 2018 00:20,306.376587,5.216037,390.900016,272.564789
3,01 01 2018 00:30,419.645905,5.659674,516.127569,271.258087
4,01 01 2018 00:40,380.650696,5.577941,491.702972,265.674286
...,...,...,...,...,...
50525,31 12 2018 23:10,2963.980957,11.404030,3397.190793,80.502724
50526,31 12 2018 23:20,1684.353027,7.332648,1173.055771,84.062599
50527,31 12 2018 23:30,2201.106934,8.435358,1788.284755,84.742500
50528,31 12 2018 23:40,2515.694092,9.421366,2418.382503,84.297913


**Clone the Libarary from Github Repo**

In [ ]:
!git clone https://github.com/thuml/Time-Series-Library.git


Cloning into 'Time-Series-Library'...
remote: Enumerating objects: 1502, done.
remote: Counting objects: 100% (755/755), done.
remote: Compressing objects: 100% (150/150), done.
remote: Total 1502 (delta 663), reused 611 (delta 605), pack-reused 747
Receiving objects: 100% (1502/1502), 78.11 MiB | 34.96 MiB/s, done.
Resolving deltas: 100% (1049/1049), done.


In [ ]:
%cd Time-Series-Library/
%ls

/content/Time-Series-Library
data_provider/  layers/  models/  README.md         run.py    tutorial/
exp/            LICENSE  pic/     requirements.txt  scripts/  utils/




>**Download the windturbine dataset in ./dataset folder as per the time series liabrary format**



In [ ]:
!gdown 'https://drive.google.com/uc?export=download&id=1fnGGCa1zmy2gNt5-lTJY8cc386V47I_l' -O ./dataset/



Downloading...
From: https://drive.google.com/uc?export=download&id=1fnGGCa1zmy2gNt5-lTJY8cc386V47I_l
To: /content/Time-Series-Library/dataset/windturbine.csv
100% 3.97M/3.97M [00:00<00:00, 231MB/s]




> **Store the dataset in correct folder for the model paths**


In [ ]:
import gdown

# URL of the Google Drive file
url = 'https://drive.google.com/uc?export=download&id=1fnGGCa1zmy2gNt5-lTJY8cc386V47I_l'

# Output path where you want to save the downloaded file
output = './dataset/ETT-small/ETTh1.csv'

# Create the dataset directory if it doesn't exist
import os
os.makedirs(os.path.dirname(output), exist_ok=True)

# Download the file
gdown.download(url, output, quiet=False)


Downloading...
From: https://drive.google.com/uc?export=download&id=1fnGGCa1zmy2gNt5-lTJY8cc386V47I_l
To: /content/Time-Series-Library/dataset/ETT-small/ETTh1.csv
100%|██████████| 3.97M/3.97M [00:00<00:00, 193MB/s]


'./dataset/ETT-small/ETTh1.csv'



> **Store the dataset in correct folder for the model paths**


In [ ]:
import gdown

# URL of the Google Drive file
url = 'https://drive.google.com/uc?export=download&id=1fnGGCa1zmy2gNt5-lTJY8cc386V47I_l'

# Output path where you want to save the downloaded file
output = './dataset/ETT-small/ETTh2.csv'

# Create the dataset directory if it doesn't exist
import os
os.makedirs(os.path.dirname(output), exist_ok=True)

# Download the file
gdown.download(url, output, quiet=False)


Downloading...
From: https://drive.google.com/uc?export=download&id=1fnGGCa1zmy2gNt5-lTJY8cc386V47I_l
To: /content/Time-Series-Library/dataset/ETT-small/ETTh2.csv
100%|██████████| 3.97M/3.97M [00:00<00:00, 195MB/s]


'./dataset/ETT-small/ETTh2.csv'



> **Convert the datetime date which is numerical instead of character format**



In [ ]:
import pandas as pd
import os

# Define the file path
file_path = './dataset/ETT-small/ETTh1.csv'

# Check if the file exists
if not os.path.exists(file_path):
    raise FileNotFoundError(f"The file '{file_path}' was not found.")

# Load the CSV file into a DataFrame
windTurbdf = pd.read_csv(file_path)

# Display the current columns to verify the column names
print("Current columns:", windTurbdf.columns)

# Define the time format
time_format = "%d %m %Y %H:%M"

# Convert the "Date/Time" column to datetime format
windTurbdf["date"] = pd.to_datetime(windTurbdf["Date/Time"], format=time_format)

# Convert the "date" column to Unix timestamps (numeric representation)
windTurbdf["date"] = windTurbdf["date"].astype(int) // 10**9  # Convert nanoseconds to seconds

# Drop the original "Date/Time" column and the "Theoretical_Power_Curve (KWh)" column
windTurbdf = windTurbdf.drop(columns=['Date/Time', 'Theoretical_Power_Curve (KWh)'])

# Rename columns to match your desired output
windTurbdf.columns = ['date', 'LV ActivePower (kW)', 'Wind Speed (m/s)', 'Wind Direction (°)']

# Define the output directory path
output_dir = './dataset/ETT-small/'

# Create the directory if it doesn't exist
os.makedirs(output_dir, exist_ok=True)

# Define the output file path
output_file_path = os.path.join(output_dir, 'ETTh1.csv')

# Save the modified DataFrame to a new CSV file
windTurbdf.to_csv(output_file_path, index=False)

print(f"Modified DataFrame saved to {output_file_path}")

# Print the columns of the modified DataFrame to verify
print("Modified DataFrame columns:", windTurbdf.columns)


Current columns: Index(['Date/Time', 'LV ActivePower (kW)', 'Wind Speed (m/s)',
       'Theoretical_Power_Curve (KWh)', 'Wind Direction (°)'],
      dtype='object')
Modified DataFrame saved to ./dataset/ETT-small/ETTh1.csv
Modified DataFrame columns: Index(['date', 'LV ActivePower (kW)', 'Wind Speed (m/s)',
       'Wind Direction (°)'],
      dtype='object')




> **Convert the datetime date which is numerical instead of character format**



In [ ]:
import pandas as pd
import os

# Define the file path
file_path = './dataset/ETT-small/ETTh2.csv'

# Check if the file exists
if not os.path.exists(file_path):
    raise FileNotFoundError(f"The file '{file_path}' was not found.")

# Load the CSV file into a DataFrame
windTurbdf = pd.read_csv(file_path)

# Display the current columns to verify the column names
print("Current columns:", windTurbdf.columns)

# Define the time format
time_format = "%d %m %Y %H:%M"

# Convert the "Date/Time" column to datetime format
windTurbdf["date"] = pd.to_datetime(windTurbdf["Date/Time"], format=time_format)

# Convert the "date" column to Unix timestamps (numeric representation)
windTurbdf["date"] = windTurbdf["date"].astype(int) // 10**9  # Convert nanoseconds to seconds

# Drop the original "Date/Time" column and the "Theoretical_Power_Curve (KWh)" column
windTurbdf = windTurbdf.drop(columns=['Date/Time', 'Theoretical_Power_Curve (KWh)'])

# Rename columns to match your desired output
windTurbdf.columns = ['date', 'LV ActivePower (kW)', 'Wind Speed (m/s)', 'Wind Direction (°)']

# Define the output directory path
output_dir = './dataset/ETT-small/'

# Create the directory if it doesn't exist
os.makedirs(output_dir, exist_ok=True)

# Define the output file path
output_file_path = os.path.join(output_dir, 'ETTh2.csv')

# Save the modified DataFrame to a new CSV file
windTurbdf.to_csv(output_file_path, index=False)

print(f"Modified DataFrame saved to {output_file_path}")

# Print the columns of the modified DataFrame to verify
print("Modified DataFrame columns:", windTurbdf.columns)


Current columns: Index(['Date/Time', 'LV ActivePower (kW)', 'Wind Speed (m/s)',
       'Theoretical_Power_Curve (KWh)', 'Wind Direction (°)'],
      dtype='object')
Modified DataFrame saved to ./dataset/ETT-small/ETTh2.csv
Modified DataFrame columns: Index(['date', 'LV ActivePower (kW)', 'Wind Speed (m/s)',
       'Wind Direction (°)'],
      dtype='object')




> ***Install relevant liabraries***



In [ ]:
!pip install patool sktime reformer_pytorch mamba_ssm


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.6/96.6 kB 1.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.0/24.0 MB 43.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.4/85.4 kB 13.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.9/132.9 kB 19.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.2/43.2 kB 7.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 307.2/307.2 kB 41.0 MB/s eta 0:00:00
  Using cached nvidia_cuda_nvrtc_cu12-12.1.105-py3-none-manylinux1_x86_64.whl (23.7 MB)
  Using cached nvidia_cuda_runtime_cu12-12.1.105-py3-none-manylinux1_x86_64.whl (823 kB)
  Using cached nvidia_cuda_cupti_cu12-12.1.105-py3-none-manylinux1_x86_64.whl (14.1 MB)
  Using cached nvidia_cudnn_cu12-8.9.2.26-py3-none-manylinux1_x86_64.whl (731.7 MB)
  Using cached nvidia_cublas_cu12-12.1.3.1-py3-none-manylinux1_x86_64.whl (410.6 


> **Set the tune parameters as number of features are 3 while the model has 7 features as default**




In [ ]:
import os

# Define the file path
script_path = './scripts/long_term_forecast/ETT_script/DLinear_ETTh1.sh'

# Check if the script file exists
if not os.path.exists(script_path):
    raise FileNotFoundError(f"The script file '{script_path}' was not found.")

# List of replacements to perform
replacements = [
    ('enc_in 7', 'enc_in 3'),
    ('dec_in 7', 'dec_in 3'),
    ('c_out 7', 'c_out 3')
]

# Read the content of the script
with open(script_path, 'r') as file:
    script_content = file.read()

# Perform all replacements
for old, new in replacements:
    script_content = script_content.replace(old, new)

# Write the modified content back to the script file
with open(script_path, 'w') as file:
    file.write(script_content)

print(f"Script '{script_path}' has been modified.")


Script './scripts/long_term_forecast/ETT_script/DLinear_ETTh1.sh' has been modified.


In [ ]:
import os

# Define the file path
script_path = './scripts/long_term_forecast/ETT_script/DLinear_ETTh1.sh'

# Check if the script file exists
if not os.path.exists(script_path):
    raise FileNotFoundError(f"The script file '{script_path}' was not found.")

# List of replacements to perform
replacements = [
    ('enc_in 7', 'enc_in 3'),
    ('dec_in 7', 'dec_in 3'),
    ('c_out 7', 'c_out 3')
]

# Read the content of the script
with open(script_path, 'r') as file:
    script_content = file.read()

# Perform all replacements
for old, new in replacements:
    script_content = script_content.replace(old, new)

# Write the modified content back to the script file
with open(script_path, 'w') as file:
    file.write(script_content)

print(f"Script '{script_path}' has been modified.")


Script './scripts/long_term_forecast/ETT_script/DLinear_ETTh1.sh' has been modified.


In [ ]:
import os

# Define the file path
script_path = './scripts/long_term_forecast/ETT_script/TimesNet_ETTh2.sh'

# Check if the script file exists
if not os.path.exists(script_path):
    raise FileNotFoundError(f"The script file '{script_path}' was not found.")

# List of replacements to perform
replacements = [
    ('enc_in 7', 'enc_in 3'),
    ('dec_in 7', 'dec_in 3'),
    ('c_out 7', 'c_out 3')
]

# Read the content of the script
with open(script_path, 'r') as file:
    script_content = file.read()

# Perform all replacements
for old, new in replacements:
    script_content = script_content.replace(old, new)

# Write the modified content back to the script file
with open(script_path, 'w') as file:
    file.write(script_content)

print(f"Script '{script_path}' has been modified.")


Script './scripts/long_term_forecast/ETT_script/TimesNet_ETTh2.sh' has been modified.


In [ ]:
import os

# Define the file path
script_path = './scripts/long_term_forecast/ETT_script/TSMixer_ETTh2.sh'

# Check if the script file exists
if not os.path.exists(script_path):
    raise FileNotFoundError(f"The script file '{script_path}' was not found.")

# List of replacements to perform
replacements = [
    ('enc_in 7', 'enc_in 3'),
    ('dec_in 7', 'dec_in 3'),
    ('c_out 7', 'c_out 3')
]

# Read the content of the script
with open(script_path, 'r') as file:
    script_content = file.read()

# Perform all replacements
for old, new in replacements:
    script_content = script_content.replace(old, new)

# Write the modified content back to the script file
with open(script_path, 'w') as file:
    file.write(script_content)

print(f"Script '{script_path}' has been modified.")




> **Start training and testing the DLinear model by using the liabrary script**



In [ ]:
!bash ./scripts/long_term_forecast/ETT_script/DLinear_ETTh1.sh

False
Args in experiment:
Basic Config
  Task Name:          long_term_forecast  Is Training:        1                   
  Model ID:           ETTh1_96_96         Model:              DLinear             

Data Loader
  Data:               ETTh1               Root Path:          ./dataset/ETT-small/
  Data Path:          ETTh1.csv           Features:           M                   
  Target:             OT                  Freq:               h                   
  Checkpoints:        ./checkpoints/      

Forecasting Task
  Seq Len:            96                  Label Len:          48                  
  Pred Len:           96                  Seasonal Patterns:  Monthly             
  Inverse:            0                   

Model Parameters
  Top k:              5                   Num Kernels:        6                   
  Enc In:             3                   Dec In:             3                   
  C Out:              3                   d model:            512              



> **Start training and testing the TSMixer model by using the liabrary script**



In [ ]:
!bash ./scripts/long_term_forecast/ETT_script/TSMixer_ETTh2.sh

False
Args in experiment:
Basic Config
  Task Name:          long_term_forecast  Is Training:        1                   
  Model ID:           ETTh2_96_96         Model:              TSMixer             

Data Loader
  Data:               ETTh2               Root Path:          ./dataset/ETT-small/
  Data Path:          ETTh2.csv           Features:           M                   
  Target:             OT                  Freq:               h                   
  Checkpoints:        ./checkpoints/      

Forecasting Task
  Seq Len:            96                  Label Len:          48                  
  Pred Len:           96                  Seasonal Patterns:  Monthly             
  Inverse:            0                   

Model Parameters
  Top k:              5                   Num Kernels:        6                   
  Enc In:             3                   Dec In:             3                   
  C Out:              3                   d model:            512              

Start Training and Testing TimesNet model

In [ ]:
!bash ./scripts/long_term_forecast/ETT_script/TimesNet_ETTh2.sh

False
Args in experiment:
Basic Config
  Task Name:          long_term_forecast  Is Training:        1                   
  Model ID:           ETTh2_96_96         Model:              TimesNet            

Data Loader
  Data:               ETTh2               Root Path:          ./dataset/ETT-small/
  Data Path:          ETTh2.csv           Features:           M                   
  Target:             OT                  Freq:               h                   
  Checkpoints:        ./checkpoints/      

Forecasting Task
  Seq Len:            96                  Label Len:          48                  
  Pred Len:           96                  Seasonal Patterns:  Monthly             
  Inverse:            0                   

Model Parameters
  Top k:              5                   Num Kernels:        6                   
  Enc In:             3                   Dec In:             3                   
  C Out:              3                   d model:            32               